# LeetCode 439 — Ternary Expression Parser bằng Python

Link đề: https://leetcode.com/problems/ternary-expression-parser/ · Độ khó: **Medium** · Chủ đề: Stack, Recursion, String

Skeleton LeetCode:

```python
class Solution:
    def parseTernary(self, expression: str) -> str:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản LeetCode Premium để xem đề gốc). Đề trong notebook được viết theo hiểu biết về bài, nên hãy **đối chiếu lại với đề gốc** nếu có tài khoản Premium; các đáp án đều đã được kiểm chứng bằng code (so với bộ sinh biểu thức có giá trị biết trước và cách rút gọn chuỗi).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính giá trị của một biểu thức **điều kiện ba ngôi lồng nhau** như `F?1:T?4:5`, trong đó `T` là đúng, `F` là sai, các số là một chữ số, và các biểu thức **kết hợp từ phải sang trái** (right-associative).
2. Nắm các công cụ Python: chỉ số và lát cắt chuỗi (`s[::-1]`), `str.rfind`, **ngăn xếp** bằng `list` (`append`/`pop`), hàm đệ quy với con trỏ vị trí, biến đếm lồng để tìm dấu `:` khớp, và `sys.setrecursionlimit`.
3. Đi từ **rút gọn chuỗi nhiều lần (tìm `?` ngoài cùng bên phải) `O(n²)`** → **đệ quy xuống theo con trỏ** → **ngăn xếp duyệt từ phải sang trái** → **ngăn xếp duyệt từ trái sang phải có nhìn trước** → **nhảy thẳng theo điều kiện `O(1)` bộ nhớ phụ**.
4. Hiểu mấu chốt: "kết hợp phải sang trái" nghĩa là **`?` ở ngoài cùng bên phải luôn có hai nhánh là giá trị đơn** (`X?a:b`), nên có thể rút gọn dần từ phải sang trái; hoặc "đọc điều kiện rồi chỉ đi vào đúng nhánh cần".
5. Tránh các lỗi kinh điển: tách theo dấu `:` **đầu tiên** (không khớp lồng), **đảo nhánh** khi chọn giá trị, rút gọn mẫu **từ trái** (nhầm điều kiện của biểu thức kế tiếp thành giá trị), và **vỡ đệ quy** với biểu thức rất sâu.

## 2. Đề bài & ví dụ

Cho chuỗi `expression` là một biểu thức điều kiện ba ngôi `điều_kiện ? nếu_đúng : nếu_sai` có thể **lồng tuỳ ý**. Hãy tính giá trị của nó và trả về dưới dạng **chuỗi một ký tự**.

Biết rằng:

- Biểu thức luôn **hợp lệ** và chỉ gồm chữ số `0..9`, ký tự `'?'`, `':'`, `'T'`, `'F'` (`T` là đúng, `F` là sai).
- Mọi số trong biểu thức đều là **một chữ số**.
- Biểu thức **kết hợp từ phải sang trái** (như hầu hết ngôn ngữ lập trình): `a?b:c?d:e` hiểu là `a?b:(c?d:e)`.
- Kết quả luôn là một chữ số, `'T'` hoặc `'F'`.

**Ví dụ 1**

- Input: `expression = "T?2:3"`
- Output: `"2"`
- Giải thích: điều kiện đúng nên chọn `2`.

**Ví dụ 2**

- Input: `expression = "F?1:T?4:5"`
- Output: `"4"`
- Giải thích: hiểu là `F ? 1 : (T ? 4 : 5)` → `F ? 1 : 4` → `4`.

**Ví dụ 3**

- Input: `expression = "T?T?F:5:3"`
- Output: `"F"`
- Giải thích: hiểu là `T ? (T ? F : 5) : 3` → `T ? F : 3` → `F`.

**Ràng buộc:**

- `5 <= expression.length <= 10^4`
- `expression` chỉ gồm chữ số, `'T'`, `'F'`, `'?'`, `':'`.
- Đảm bảo `expression` là biểu thức ba ngôi hợp lệ và mỗi số chỉ có một chữ số.

Chú ý: **điều kiện** luôn là `T` hoặc `F` (một ký tự), nhưng **giá trị trả về** của một nhánh có thể cũng là `T`/`F` hoặc chữ số, hoặc một biểu thức lồng.

In [1]:
import sys
import random

examples = [
    ("T?2:3", "2"),
    ("F?1:T?4:5", "4"),
    ("T?T?F:5:3", "F"),
]
for expr, expected in examples:
    print(repr(expr), "-> kỳ vọng", repr(expected))

'T?2:3' -> kỳ vọng '2'
'F?1:T?4:5' -> kỳ vọng '4'
'T?T?F:5:3' -> kỳ vọng 'F'


## 3. Kiến thức nền

### 3.1. Biểu thức điều kiện `a if cond else b` trong Python

Python viết "nếu `cond` thì `a` ngược lại `b`" là `a if cond else b`. Biểu thức ba ngôi của đề (`cond ? a : b`) có đúng nghĩa này. Ví dụ `"T?2:3"` ứng với `2 if True else 3`.

In [2]:
def pick(cond_char, a, b):
    return a if cond_char == "T" else b


print(pick("T", "2", "3"), pick("F", "2", "3"))      # 2 3

2 3


### 3.2. Kết hợp phải sang trái (right-associative)

Với `F?1:T?4:5`, có hai cách đặt ngoặc: `(F?1:T)?4:5` hoặc `F?1:(T?4:5)`. Đề quy định cách **thứ hai**: dấu `?` **bên phải** liên kết trước. Hệ quả hữu ích: **ở `?` ngoài cùng bên phải, cả hai nhánh `a`, `b` đều là giá trị đơn** (một ký tự) vì phía sau nó không còn `?` nào nữa.

Ví dụ `T?T?F:5:3`: `?` cuối là ở vị trí 3; xung quanh nó là `T ? F : 5` — rút gọn thành `F` rồi biểu thức còn `T?F:3`.

In [3]:
expr = "T?T?F:5:3"
k = expr.rfind("?")                      # vị trí của '?' ngoài cùng bên phải
print(k, expr[k - 1], expr[k + 1], expr[k + 2], expr[k + 3])     # 3 T F : 5
print(expr[k - 1:k + 4])                                         # T?F:5

3 T F : 5
T?F:5


### 3.3. Chỉ số, lát cắt và `str.rfind`

`s[i]` lấy ký tự thứ `i`; `s[i:j]` lấy đoạn `[i, j)`; `s[::-1]` là chuỗi đảo ngược. `s.rfind(x)` trả về vị trí **xuất hiện cuối cùng** của `x` (hoặc `-1` nếu không có); còn `s.find(x)` trả vị trí **đầu tiên**.

In [4]:
s = "F?1:T?4:5"
print(s[::-1])                         # 5:4?T:1?F
print(s.find("?"), s.rfind("?"))       # 1 5
print(s.find(":"), s.rfind(":"))       # 3 7
print(s[:3] + "X" + s[5:])             # ghép chuỗi mới từ các lát cắt

5:4?T:1?F
1 5
3 7
F?1X?4:5


### 3.3b. Ngăn xếp (stack) bằng `list`

Ngăn xếp: vào sau ra trước. `stk.append(x)` đẩy vào đỉnh, `stk.pop()` lấy đỉnh ra, `stk[-1]` nhìn đỉnh không lấy ra. Ta cần ngăn xếp để "nhớ" các giá trị đã đọc nhưng **chưa dùng đến** khi gặp điều kiện.

In [5]:
stk = []
stk.append("a")
stk.append("b")
stk.append("c")
print(stk, stk[-1])        # ['a', 'b', 'c'] c
print(stk.pop(), stk)      # c ['a', 'b']
print(stk.pop(), stk.pop(), stk)    # b a []

['a', 'b', 'c'] c
c ['a', 'b']
b a []


### 3.4. Đếm lồng để tìm dấu `:` khớp với một dấu `?`

Trong `T?T?F:5:3`, dấu `?` đầu tiên **khớp** với dấu `:` cuối cùng (không phải dấu `:` đầu tiên). Cách tìm: sau dấu `?`, đặt `depth = 1`; duyệt tiếp: gặp `?` thì `depth += 1`, gặp `:` thì `depth -= 1`; khi `depth` về `0`, ta tìm được dấu `:` khớp. Giống hệt cách khớp ngoặc `(` `)`.

In [6]:
def matching_colon(expr, q):
    """q là vị trí một dấu '?'. Trả về vị trí dấu ':' khớp với nó."""
    depth = 1
    for j in range(q + 1, len(expr)):
        if expr[j] == "?":
            depth += 1
        elif expr[j] == ":":
            depth -= 1
            if depth == 0:
                return j
    return -1


e = "T?T?F:5:3"
print(matching_colon(e, 1), e.find(":"))     # 7 5   (khớp thật là 7, dấu ':' đầu tiên là 5)

7 5


### 3.5. Đệ quy với con trỏ vị trí và giới hạn đệ quy

Hàm đệ quy `parse()` đọc biểu thức bằng một biến vị trí `pos` dùng chung (qua `nonlocal`/list): đọc xong phần nào thì `pos` tiến tới phần đó. Python mặc định cho phép độ sâu đệ quy khoảng `1000`; biểu thức lồng sâu có thể vượt (xem mục lỗi thường gặp). `sys.setrecursionlimit(n)` nâng giới hạn này.

In [7]:
print(sys.getrecursionlimit())          # thường là 1000

def depth_demo(k):
    return 0 if k == 0 else 1 + depth_demo(k - 1)

print(depth_demo(500))
try:
    depth_demo(5000)
except RecursionError as e:
    print("RecursionError:", str(e)[:40])

1000
500
RecursionError: maximum recursion depth exceeded


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính `F?1:T?4:5` bằng cách "viết ngoặc rồi rút gọn" như đề gợi ý:

| Bước | Biểu thức | Làm gì |
|---|---|---|
| 0 | `F?1:T?4:5` | ban đầu |
| 1 | `F?1:(T?4:5)` | thêm ngoặc theo luật phải-sang-trái |
| 2 | `F?1:4` | rút gọn trong ngoặc: `T` đúng nên chọn `4` |
| 3 | `4` | `F` sai nên chọn nhánh sau: `4` |

Còn `T?T?F:5:3` (`T ? (T ? F : 5) : 3`):

| Bước | Biểu thức | Làm gì |
|---|---|---|
| 0 | `T?T?F:5:3` | ban đầu |
| 1 | `T?F:3` | rút gọn trong cùng `T?F:5` → `F` (vì `T` đúng chọn `F`) |
| 2 | `F` | `T` đúng chọn nhánh trước = `F` |

Quy tắc chung: **luôn rút gọn dấu `?` ngoài cùng bên phải**, vì hai nhánh của nó đều đã là giá trị đơn.

In [8]:
expr = "F?1:T?4:5"
while "?" in expr:
    k = expr.rfind("?")
    cond, a, b = expr[k - 1], expr[k + 1], expr[k + 3]
    value = a if cond == "T" else b
    new = expr[:k - 1] + value + expr[k + 4:]
    print(f"{expr:>12}  --[{cond}?{a}:{b} = {value}]-->  {new}")
    expr = new
print("kết quả:", expr)

   F?1:T?4:5  --[T?4:5 = 4]-->  F?1:4
       F?1:4  --[F?1:4 = 4]-->  4
kết quả: 4


## 5. Cách 1 — Brute Force / baseline: rút gọn chuỗi lặp đi lặp lại

**Ý tưởng.** Đúng như bảng tính tay: lặp đến khi hết `?`: tìm `?` **cuối cùng** ở vị trí `k`; ba ký tự quanh nó và dấu `:` tạo thành `s[k-1] ? s[k+1] : s[k+3]`; thay đoạn `s[k-1 : k+4]` (5 ký tự) bằng giá trị được chọn.

In [9]:
def parse_rewrite(expression):
    s = expression
    while "?" in s:
        k = s.rfind("?")
        cond, a, b = s[k - 1], s[k + 1], s[k + 3]
        value = a if cond == "T" else b
        s = s[:k - 1] + value + s[k + 4:]
    return s


for expr, expected in examples:
    print(parse_rewrite(expr), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
F (kỳ vọng F )


**Độ phức tạp (Cách 1).** Mỗi lần rút gọn bớt `4` ký tự nên có tối đa `n/4` vòng; mỗi vòng `rfind` + dựng chuỗi mới tốn `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`** (chuỗi mới). Với `n = 10^4` còn chấp nhận được (`~ 2.5·10^7` ký tự sao chép), nhưng đây là cách đơn giản nhất chứ chưa tối ưu.

**Nút thắt:** mỗi vòng ta **sao chép lại gần cả chuỗi** chỉ để thay 5 ký tự. Nếu dùng **ngăn xếp** thì việc "thay thế" chỉ là `pop`/`append` ở đỉnh, tốn `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đọc từ phải sang trái và dùng ngăn xếp.** Tại sao phải sang trái? Vì khi gặp `?` ngoài cùng bên phải, hai giá trị `a` và `b` của nó **vừa được đọc xong** (chúng đứng ngay bên phải `?`) và đang nằm ở đỉnh ngăn xếp.

Duyệt `expression` từ **phải sang trái**:

1. Gặp `:` → bỏ qua (chỉ là dấu phân cách).
2. Gặp `?` → đánh dấu "ký tự kế tiếp (bên trái) là **điều kiện**".
3. Gặp ký tự khác `c`:
   - Nếu đang chờ điều kiện (vừa gặp `?`): `c` là điều kiện. Hai phần tử trên cùng của ngăn xếp là `a` (đỉnh, đọc sau cùng nên nằm gần `?` nhất = nhánh **đúng**) và `b` (dưới đỉnh = nhánh **sai**). Nếu `c == 'T'` giữ `a`, ngược lại giữ `b`; đẩy giá trị giữ lại vào ngăn xếp.
   - Ngược lại `c` là một **giá trị**: đẩy vào ngăn xếp.
4. Cuối cùng ngăn xếp còn đúng một phần tử: đáp án.

Vì sao "đỉnh = nhánh đúng"? Duyệt từ phải sang trái thì `b` (bên phải) vào ngăn xếp trước, rồi `a` vào sau nên `a` ở trên đỉnh.

In [10]:
expr = "T?T?F:5:3"
stk, cond = [], False
for c in expr[::-1]:
    if c == ":":
        continue
    if c == "?":
        cond = True
        continue
    if cond:
        a = stk.pop()
        b = stk.pop()
        stk.append(a if c == "T" else b)
        print(f"điều kiện {c}: a={a}, b={b} -> giữ {stk[-1]}   ngăn xếp = {stk}")
        cond = False
    else:
        stk.append(c)
        print(f"giá trị  {c}:                      ngăn xếp = {stk}")
print("kết quả:", stk[0])

giá trị  3:                      ngăn xếp = ['3']
giá trị  5:                      ngăn xếp = ['3', '5']
giá trị  F:                      ngăn xếp = ['3', '5', 'F']
điều kiện T: a=F, b=5 -> giữ F   ngăn xếp = ['3', 'F']
điều kiện T: a=F, b=3 -> giữ F   ngăn xếp = ['F']
kết quả: F


## 7. Các cách tiếp theo

### Cách 2 — Đệ quy xuống theo con trỏ (recursive descent)

**Giải thích từng đoạn.**

1. Giữ `pos` = vị trí ký tự sắp đọc.
2. `parse()`: đọc `c = s[pos]` (và tăng `pos`). Nếu **ký tự kế tiếp không phải `?`** (hoặc hết chuỗi) thì `c` là một **giá trị đơn**: trả `c`.
3. Ngược lại `c` là điều kiện: bỏ qua `?`; `a = parse()` (nhánh đúng); bỏ qua `:`; `b = parse()` (nhánh sai); trả `a` nếu `c == 'T'`, ngược lại `b`.

Đệ quy này **tính cả hai nhánh** rồi mới chọn (đơn giản, vẫn `O(n)` vì mỗi ký tự được đọc một lần).

In [11]:
def parse_recursive(expression):
    s = expression
    pos = 0

    def parse():
        nonlocal pos
        c = s[pos]
        pos += 1
        if pos >= len(s) or s[pos] != "?":
            return c                      # giá trị đơn (kết thúc nhánh)
        pos += 1                          # bỏ qua '?'
        a = parse()                       # nhánh đúng
        pos += 1                          # bỏ qua ':'
        b = parse()                       # nhánh sai
        return a if c == "T" else b

    return parse()


for expr, expected in examples:
    print(parse_recursive(expr), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
F (kỳ vọng F )


In [12]:
# Code nộp LeetCode cho Cách 2 (cần nâng giới hạn đệ quy nếu biểu thức lồng rất sâu)
class SolutionRecursive:
    def parseTernary(self, expression: str) -> str:
        sys.setrecursionlimit(20000)
        pos = 0

        def parse():
            nonlocal pos
            c = expression[pos]
            pos += 1
            if pos >= len(expression) or expression[pos] != "?":
                return c
            pos += 1
            a = parse()
            pos += 1
            b = parse()
            return a if c == "T" else b

        return parse()


print(SolutionRecursive().parseTernary("T?T?F:5:3"))      # F

F


**Độ phức tạp Cách 2:** mỗi ký tự được đọc đúng một lần ⇒ **Time `O(n)`**; **Space `O(d)`** với `d` là độ sâu lồng (tối đa khoảng `n/4`, cỡ `2500` với `n = 10^4`) cho ngăn xếp đệ quy. Độ sâu này vượt giới hạn mặc định `1000` của Python nên phải `setrecursionlimit`.

### Cách 3 — Ngăn xếp duyệt từ phải sang trái (cách chuẩn)

**Giải thích từng đoạn.** (xem mục 6)

1. Duyệt `expression[::-1]`; `cond` = cờ "ký tự kế tiếp là điều kiện".
2. `':'` bỏ qua; `'?'` đặt `cond = True`.
3. Ký tự `c` còn lại: nếu `cond` thì `a = stk.pop()`, `b = stk.pop()`, đẩy `a` nếu `c == 'T'` ngược lại `b`; không thì đẩy `c`.
4. Trả `stk[0]`.

In [13]:
def parse_stack_rtl(expression):
    stk = []
    cond = False
    for c in expression[::-1]:
        if c == ":":
            continue
        if c == "?":
            cond = True
        elif cond:
            a = stk.pop()                 # nhánh đúng (đỉnh)
            b = stk.pop()                 # nhánh sai
            stk.append(a if c == "T" else b)
            cond = False
        else:
            stk.append(c)
    return stk[0]


for expr, expected in examples:
    print(parse_stack_rtl(expr), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
F (kỳ vọng F )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionStackRTL:
    def parseTernary(self, expression: str) -> str:
        stk = []
        cond = False
        for c in expression[::-1]:
            if c == ":":
                continue
            if c == "?":
                cond = True
            else:
                if cond:
                    if c == "T":
                        x = stk.pop()
                        stk.pop()
                        stk.append(x)
                    else:
                        stk.pop()
                    cond = False
                else:
                    stk.append(c)
        return stk[0]


print(SolutionStackRTL().parseTernary("F?1:T?4:5"))      # 4

4


**Độ phức tạp Cách 3:** mỗi ký tự được xử lý một lần với `O(1)` thao tác ngăn xếp ⇒ **Time `O(n)`**, **Space `O(n)`** (ngăn xếp có thể chứa tới khoảng `n/2` ký tự giá trị trong trường hợp xấu).

### Cách 4 — Ngăn xếp duyệt từ trái sang phải, rút gọn khi đủ mẫu (có nhìn trước)

**Giải thích từng đoạn.**

1. Đẩy lần lượt từng ký tự vào ngăn xếp.
2. Sau khi đẩy một **giá trị** (không phải `?`, `:`), kiểm tra **ký tự kế tiếp** của biểu thức: nếu nó là `'?'` thì giá trị vừa đẩy thực chất là **điều kiện** của một biểu thức sắp mở, **không** rút gọn. Nếu không, thử rút gọn: khi đỉnh ngăn xếp có dạng `X ? a : b` (tức `stk[-2] == ':'` và `stk[-4] == '?'`) thì thay 5 ký tự đó bằng giá trị được chọn, và **lặp lại** (có thể rút gọn tiếp lớp ngoài).
3. Cuối cùng ngăn xếp còn đúng một ký tự.

"Nhìn trước" ký tự kế tiếp là điểm mấu chốt để không nhầm điều kiện (như `T` trong `F?1:T?4:5`) thành nhánh sai `b`.

In [15]:
def parse_stack_ltr(expression):
    stk = []
    n = len(expression)
    for i, c in enumerate(expression):
        stk.append(c)
        if c in "?:":
            continue
        if i + 1 < n and expression[i + 1] == "?":
            continue                                    # c là điều kiện của biểu thức kế tiếp
        while len(stk) >= 5 and stk[-2] == ":" and stk[-4] == "?":
            b = stk.pop()
            stk.pop()                                   # ':'
            a = stk.pop()
            stk.pop()                                   # '?'
            cond = stk.pop()
            stk.append(a if cond == "T" else b)
    return stk[0]


for expr, expected in examples:
    print(parse_stack_ltr(expr), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
F (kỳ vọng F )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionStackLTR:
    def parseTernary(self, expression: str) -> str:
        stk = []
        n = len(expression)
        for i, c in enumerate(expression):
            stk.append(c)
            if c in "?:":
                continue
            if i + 1 < n and expression[i + 1] == "?":
                continue
            while len(stk) >= 5 and stk[-2] == ":" and stk[-4] == "?":
                b = stk.pop()
                stk.pop()
                a = stk.pop()
                stk.pop()
                cond = stk.pop()
                stk.append(a if cond == "T" else b)
        return stk[0]


print(SolutionStackLTR().parseTernary("T?T?F:5:3"))      # F

F


**Độ phức tạp Cách 4:** mỗi ký tự đẩy vào một lần và mỗi lần rút gọn bớt `4` ký tự khỏi ngăn xếp, nên tổng số lần `pop`/`append` là `O(n)` ⇒ **Time `O(n)`**, **Space `O(n)`**.

### Cách 5 — Nhảy thẳng theo điều kiện: chỉ đi vào nhánh cần (`O(1)` bộ nhớ phụ)

**Giải thích từng đoạn.** Ta không cần tính cả hai nhánh — chỉ **đi vào nhánh được chọn** và **bỏ qua nhánh kia**.

1. `i = 0`. Lặp: nếu `i + 1 >= n` hoặc `expression[i + 1] != '?'` thì `expression[i]` là một **giá trị đơn** và đó là đáp án: trả về.
2. Ngược lại `expression[i]` là điều kiện, `expression[i + 1] == '?'`.
   - Nếu là `'T'`: đi vào nhánh đúng, `i += 2` (nhảy qua điều kiện và `?`). Nhánh đúng kết thúc ở dấu `:` khớp nhưng ta không cần quan tâm: lát nữa gặp giá trị đơn là dừng luôn.
   - Nếu là `'F'`: bỏ qua nhánh đúng bằng cách **đếm lồng** (mục 3.4) để tìm dấu `:` khớp với dấu `?` này, rồi `i = (vị trí dấu ':') + 1` (bắt đầu nhánh sai).

Con trỏ `i` chỉ đi tới, mỗi ký tự bị quét qua tối đa một lần (khi nhảy qua) ⇒ tuyến tính.

In [17]:
def parse_jump(expression):
    n = len(expression)
    i = 0
    while True:
        if i + 1 >= n or expression[i + 1] != "?":
            return expression[i]                        # giá trị đơn
        if expression[i] == "T":
            i += 2                                      # vào nhánh đúng
        else:
            depth = 1                                   # nhảy qua nhánh đúng
            j = i + 2
            while depth:
                if expression[j] == "?":
                    depth += 1
                elif expression[j] == ":":
                    depth -= 1
                j += 1
            i = j                                       # j đã đứng sau dấu ':' khớp


for expr, expected in examples:
    print(parse_jump(expr), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
F (kỳ vọng F )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionJump:
    def parseTernary(self, expression: str) -> str:
        n = len(expression)
        i = 0
        while True:
            if i + 1 >= n or expression[i + 1] != "?":
                return expression[i]
            if expression[i] == "T":
                i += 2
            else:
                depth = 1
                j = i + 2
                while depth:
                    if expression[j] == "?":
                        depth += 1
                    elif expression[j] == ":":
                        depth -= 1
                    j += 1
                i = j


print(SolutionJump().parseTernary("F?1:T?4:5"))      # 4

4


**Độ phức tạp Cách 5:** `i` chỉ tăng; phần "nhảy qua nhánh" cũng chỉ duyệt các ký tự phía trước `i` mới ⇒ **Time `O(n)`**, **Space `O(1)`** (không ngăn xếp, không đệ quy). Đây là cách tiết kiệm bộ nhớ nhất.

## 8. TRACE từng bước

Ta trace Cách 3 (ngăn xếp phải→trái) trên `T?T?F:5:3`, Cách 4 (trái→phải có nhìn trước) trên `F?1:T?4:5`, và Cách 5 (nhảy) trên `F?T?1:2:T?3:4`.

In [19]:
def trace_stack_rtl(expression):
    stk, cond = [], False
    print("biểu thức:", expression, "| duyệt ngược:", expression[::-1])
    for idx in range(len(expression) - 1, -1, -1):
        c = expression[idx]
        if c == ":":
            print(f"i={idx:>2} {c!r}: bỏ qua dấu ':'                       stk={stk}")
            continue
        if c == "?":
            cond = True
            print(f"i={idx:>2} {c!r}: ký tự kế tiếp là điều kiện            stk={stk}")
            continue
        if cond:
            a, b = stk.pop(), stk.pop()
            stk.append(a if c == "T" else b)
            cond = False
            print(f"i={idx:>2} {c!r}: điều kiện, a={a} b={b} -> giữ {stk[-1]!r}     stk={stk}")
        else:
            stk.append(c)
            print(f"i={idx:>2} {c!r}: giá trị, đẩy vào                      stk={stk}")
    return stk[0]


assert trace_stack_rtl("T?T?F:5:3") == "F"

biểu thức: T?T?F:5:3 | duyệt ngược: 3:5:F?T?T
i= 8 '3': giá trị, đẩy vào                      stk=['3']
i= 7 ':': bỏ qua dấu ':'                       stk=['3']
i= 6 '5': giá trị, đẩy vào                      stk=['3', '5']
i= 5 ':': bỏ qua dấu ':'                       stk=['3', '5']
i= 4 'F': giá trị, đẩy vào                      stk=['3', '5', 'F']
i= 3 '?': ký tự kế tiếp là điều kiện            stk=['3', '5', 'F']
i= 2 'T': điều kiện, a=F b=5 -> giữ 'F'     stk=['3', 'F']
i= 1 '?': ký tự kế tiếp là điều kiện            stk=['3', 'F']
i= 0 'T': điều kiện, a=F b=3 -> giữ 'F'     stk=['F']


In [20]:
def trace_stack_ltr(expression):
    stk = []
    n = len(expression)
    print("biểu thức:", expression)
    for i, c in enumerate(expression):
        stk.append(c)
        msg = "đẩy"
        if c not in "?:":
            if i + 1 < n and expression[i + 1] == "?":
                msg = "đẩy (là điều kiện của biểu thức kế, không rút gọn)"
            else:
                while len(stk) >= 5 and stk[-2] == ":" and stk[-4] == "?":
                    b = stk.pop(); stk.pop(); a = stk.pop(); stk.pop(); cond = stk.pop()
                    stk.append(a if cond == "T" else b)
                    msg += f" + rút gọn {cond}?{a}:{b} -> {stk[-1]}"
        print(f"i={i:>2} {c!r}: {msg:<58} stk={''.join(stk)}")
    return stk[0]


assert trace_stack_ltr("F?1:T?4:5") == "4"

biểu thức: F?1:T?4:5
i= 0 'F': đẩy (là điều kiện của biểu thức kế, không rút gọn)         stk=F
i= 1 '?': đẩy                                                        stk=F?
i= 2 '1': đẩy                                                        stk=F?1
i= 3 ':': đẩy                                                        stk=F?1:
i= 4 'T': đẩy (là điều kiện của biểu thức kế, không rút gọn)         stk=F?1:T
i= 5 '?': đẩy                                                        stk=F?1:T?
i= 6 '4': đẩy                                                        stk=F?1:T?4
i= 7 ':': đẩy                                                        stk=F?1:T?4:
i= 8 '5': đẩy + rút gọn T?4:5 -> 4 + rút gọn F?1:4 -> 4              stk=4


In [21]:
def trace_jump(expression):
    n = len(expression)
    i = 0
    print("biểu thức:", expression)
    while True:
        if i + 1 >= n or expression[i + 1] != "?":
            print(f"i={i}: {expression[i]!r} là giá trị đơn -> kết quả")
            return expression[i]
        if expression[i] == "T":
            print(f"i={i}: điều kiện 'T' -> vào nhánh đúng, i = {i + 2}")
            i += 2
        else:
            depth, j = 1, i + 2
            while depth:
                depth += (expression[j] == "?") - (expression[j] == ":")
                j += 1
            print(f"i={i}: điều kiện 'F' -> nhảy qua nhánh đúng {expression[i + 2:j - 1]!r}, i = {j}")
            i = j


assert trace_jump("F?T?1:2:T?3:4") == "3"

biểu thức: F?T?1:2:T?3:4
i=0: điều kiện 'F' -> nhảy qua nhánh đúng 'T?1:2', i = 8
i=8: điều kiện 'T' -> vào nhánh đúng, i = 10
i=10: '3' là giá trị đơn -> kết quả


Bảng trace Cách 3 với `T?T?F:5:3` (duyệt từ phải sang trái; `stk` là ngăn xếp, đáy bên trái):

| `i` | Ký tự | Vai trò | Hành động | `stk` sau |
|---|---|---|---|---|
| 8 | `3` | giá trị | đẩy | `[3]` |
| 7 | `:` | phân cách | bỏ qua | `[3]` |
| 6 | `5` | giá trị | đẩy | `[3, 5]` |
| 5 | `:` | phân cách | bỏ qua | `[3, 5]` |
| 4 | `F` | giá trị | đẩy | `[3, 5, F]` |
| 3 | `?` | dấu hỏi | `cond = True` | `[3, 5, F]` |
| 2 | `T` | điều kiện | `a = F`, `b = 5`, `T` ⇒ giữ `F` | `[3, F]` |
| 1 | `?` | dấu hỏi | `cond = True` | `[3, F]` |
| 0 | `T` | điều kiện | `a = F`, `b = 3`, `T` ⇒ giữ `F` | `[F]` |

Kết quả `"F"`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (ngăn xếp từ phải sang trái): ngắn, không đệ quy nên không lo vỡ độ sâu, `O(n)`.

In [22]:
class Solution:
    def parseTernary(self, expression: str) -> str:
        stk = []
        cond = False                          # đang chờ ký tự điều kiện?
        for c in expression[::-1]:
            if c == ":":
                continue                      # chỉ là dấu phân cách
            if c == "?":
                cond = True                   # ký tự kế tiếp (bên trái) là điều kiện
            elif cond:
                a = stk.pop()                 # nhánh đúng (đỉnh)
                b = stk.pop()                 # nhánh sai
                stk.append(a if c == "T" else b)
                cond = False
            else:
                stk.append(c)                 # một giá trị
        return stk[0]


sol = Solution()
for expr, expected in examples:
    print(repr(expr), "->", repr(sol.parseTernary(expr)))

'T?2:3' -> '2'
'F?1:T?4:5' -> '4'
'T?T?F:5:3' -> 'F'


## 10. Vì sao thuật toán đúng

- **Cú pháp:** vì kết hợp phải-sang-trái, ngữ pháp là `E := leaf | cond ? E : E` với `cond ∈ {T, F}` và `leaf` là một ký tự giá trị. Mọi biểu thức hợp lệ sinh ra từ ngữ pháp này.
- **Cách 1 (rút gọn):** `?` ngoài cùng bên phải không có `?` nào ở sau nên hai nhánh là hai `leaf`; rút gọn `X?a:b` thành giá trị được chọn cho ra một biểu thức hợp lệ ngắn hơn có **cùng giá trị**. Quy nạp theo số `?` ⇒ đúng.
- **Cách 3 (ngăn xếp phải→trái):** khi gặp `?` thì phần bên phải của nó (hai biểu thức con `a` và `b`) đã được rút gọn thành **hai giá trị đơn** nằm ở đỉnh ngăn xếp (`a` trên `b`); ký tự bên trái `?` là điều kiện. Chọn một trong hai và đẩy lại: đúng như rút gọn từ phải sang trái. Kết thúc còn một phần tử vì mỗi bước điều kiện biến ba mục (`a`, `b`, điều kiện) thành một.
- **Cách 4 (trái→phải):** chỉ rút gọn khi đỉnh có dạng `X ? a : b` và ký tự kế tiếp **không phải `?`** (nghĩa là `b` là **giá trị đơn thật**, chứ không phải điều kiện của một biểu thức con sắp mở); lặp rút gọn để các lớp lồng bên ngoài được xử lý.
- **Cách 2 và 5:** theo định nghĩa đệ quy của ngữ pháp: biểu thức = điều kiện, rồi chọn nhánh. Cách 5 bỏ qua nhánh không chọn bằng khớp `?`/`:` (cùng nguyên lý khớp ngoặc), và dừng khi gặp một giá trị đơn (ký tự không theo sau bởi `?`).
- **Đối chiếu độc lập:** mục 12 sinh biểu thức từ **cây biểu thức có giá trị biết trước** (bộ sinh không dùng bất cứ parser nào ở trên), rồi so tất cả các cách.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(expression)`, `d` là độ sâu lồng tối đa (`d ≤ n/4`).

| Cách | Time | Space |
|---|---|---|
| 1. Rút gọn chuỗi lặp lại | `O(n²)` | `O(n)` |
| 2. Đệ quy theo con trỏ | `O(n)` | `O(d)` (ngăn xếp đệ quy) |
| 3. Ngăn xếp phải→trái | `O(n)` | `O(n)` |
| 4. Ngăn xếp trái→phải | `O(n)` | `O(n)` |
| 5. Nhảy theo điều kiện | `O(n)` | `O(1)` |

Với `n = 10^4`: các Cách 2–5 chỉ cỡ vài chục nghìn thao tác; Cách 1 cỡ `2.5·10^7` thao tác sao chép ký tự (vẫn nhanh vì là phép cắt chuỗi của C).

## 12. Bộ test

Vì kết quả của mỗi biểu thức là xác định, ta so với giá trị kỳ vọng. Ta còn xây **bộ sinh biểu thức** độc lập: sinh cây (lá hoặc `cond ? trái : phải`), in thành chuỗi và **tính giá trị trực tiếp trên cây**, không đi qua parser nào.

In [23]:
impls = {
    "rewrite": parse_rewrite,
    "recursive": parse_recursive,
    "stack_rtl": parse_stack_rtl,
    "stack_ltr": parse_stack_ltr,
    "jump": parse_jump,
    "leetcode": lambda e: Solution().parseTernary(e),
    "submit_recursive": lambda e: SolutionRecursive().parseTernary(e),
    "submit_rtl": lambda e: SolutionStackRTL().parseTernary(e),
    "submit_ltr": lambda e: SolutionStackLTR().parseTernary(e),
    "submit_jump": lambda e: SolutionJump().parseTernary(e),
}


def check_all(expression, expected, label):
    for key, fn in impls.items():
        actual = fn(expression)
        assert actual == expected, (label, key, expression, actual, expected)
    print("PASS", label)


for idx, (expr, expected) in enumerate(examples, 1):
    check_all(expr, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [24]:
check_all("T?1:2", "1", "biểu thức nhỏ nhất, điều kiện đúng")
check_all("F?1:2", "2", "biểu thức nhỏ nhất, điều kiện sai")
check_all("T?T:F", "T", "kết quả là T")
check_all("F?T:F", "F", "kết quả là F")
check_all("T?F:T", "F", "điều kiện T, kết quả F")
check_all("F?F:F?F:T", "T", "chuỗi nhánh sai")
check_all("F?1:F?2:F?3:4", "4", "chuỗi `else if` dài")
check_all("F?1:F?2:T?3:4", "3", "dừng ở giữa chuỗi else-if")
check_all("T?T?T?1:2:3:4", "1", "lồng sâu ở nhánh đúng")
check_all("T?F?1:2:3", "2", "nhánh đúng là biểu thức có điều kiện sai")
check_all("F?T?1:2:T?3:4", "3", "cả hai nhánh đều là biểu thức con")
check_all("T?T?F:5:3", "F", "ví dụ 3 (nhánh đúng lồng)")
check_all("F?T?1:2:F?3:T", "T", "kết quả T từ nhánh lồng")
check_all("T?9:0", "9", "chữ số 9")
check_all("F?9:0", "0", "chữ số 0")

PASS biểu thức nhỏ nhất, điều kiện đúng
PASS biểu thức nhỏ nhất, điều kiện sai
PASS kết quả là T
PASS kết quả là F
PASS điều kiện T, kết quả F
PASS chuỗi nhánh sai
PASS chuỗi `else if` dài
PASS dừng ở giữa chuỗi else-if
PASS lồng sâu ở nhánh đúng
PASS nhánh đúng là biểu thức có điều kiện sai
PASS cả hai nhánh đều là biểu thức con
PASS ví dụ 3 (nhánh đúng lồng)
PASS kết quả T từ nhánh lồng
PASS chữ số 9
PASS chữ số 0


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) tìm dấu `:` **đầu tiên** thay vì dấu `:` **khớp**; (b) **đảo nhánh** khi chọn (nhánh sai khi điều kiện đúng).

In [25]:
def wrong_first_colon(expression):
    def go(e):
        if len(e) == 1:
            return e
        cond = e[0]
        colon = e.find(":")                  # SAI: ':' đầu tiên có thể thuộc biểu thức con
        a, b = e[2:colon], e[colon + 1:]
        return go(a) if cond == "T" else go(b)
    return go(expression)


def wrong_swapped(expression):
    stk, cond = [], False
    for c in expression[::-1]:
        if c == ":":
            continue
        if c == "?":
            cond = True
        elif cond:
            a, b = stk.pop(), stk.pop()
            stk.append(b if c == "T" else a)      # SAI: đảo nhánh
            cond = False
        else:
            stk.append(c)
    return stk[0]


try:
    print("':' đầu tiên:", wrong_first_colon("T?T?F:5:3"), "| đúng:", Solution().parseTernary("T?T?F:5:3"))
except Exception as e:
    print("':' đầu tiên: lỗi", type(e).__name__, "| đúng:", Solution().parseTernary("T?T?F:5:3"))
print("đảo nhánh:", wrong_swapped("T?2:3"), "| đúng:", Solution().parseTernary("T?2:3"))
assert wrong_swapped("T?2:3") != "2"

':' đầu tiên: lỗi IndexError | đúng: F
đảo nhánh: 3 | đúng: 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) với bộ sinh độc lập

Ta sinh cây biểu thức: lá là một ký tự trong `"0123456789TF"`; nút trong là `(điều_kiện ∈ {T,F}, trái, phải)`. Chuỗi biểu thức là `cond + "?" + str(trái) + ":" + str(phải)`; giá trị tính **trực tiếp trên cây**. Ta kiểm tra độ dài `5 <= len <= 10^4` rồi so với mọi cách. Thêm hai dạng cực đoan: nhánh đúng lồng sâu (`T?T?T?...`) và chuỗi else-if sâu (`F?1:F?1:...`), để kiểm tra giới hạn đệ quy.

In [26]:
rng = random.Random(439)
LEAVES = "0123456789TF"


def gen_tree(depth_left):
    """Trả về (chuỗi, giá trị) của một biểu thức sinh ngẫu nhiên."""
    if depth_left == 0 or rng.random() < 0.25:
        c = rng.choice(LEAVES)
        return c, c
    cond = rng.choice("TF")
    s1, v1 = gen_tree(depth_left - 1)
    s2, v2 = gen_tree(depth_left - 1)
    return f"{cond}?{s1}:{s2}", (v1 if cond == "T" else v2)


tested = 0
for trial in range(4000):
    expr, value = gen_tree(rng.randint(1, 7))
    if len(expr) < 5:
        continue                                        # đề yêu cầu len >= 5
    assert len(expr) <= 10**4
    for key, fn in impls.items():
        assert fn(expr) == value, (key, expr, value)
    tested += 1
print("PASS", tested, "biểu thức ngẫu nhiên (đối chiếu với giá trị tính trên cây)")

sys.setrecursionlimit(20000)
k = 2499                                                # 4k + 1 = 9997 <= 10^4
deep_true = "T?" * k + "1" + ":2" * k                   # nhánh đúng lồng k tầng
deep_false = "F?1:" * k + "7"                           # chuỗi else-if k tầng
for name, e, v in [("lồng nhánh đúng", deep_true, "1"), ("chuỗi else-if", deep_false, "7")]:
    assert 5 <= len(e) <= 10**4
    for key, fn in impls.items():
        assert fn(e) == v, (name, key)
    print("PASS", name, "độ dài", len(e))

PASS 3021 biểu thức ngẫu nhiên (đối chiếu với giá trị tính trên cây)


PASS lồng nhánh đúng độ dài 9997
PASS chuỗi else-if độ dài 9997


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tách theo dấu `:` đầu tiên (không khớp lồng)

In [27]:
def sai_dau_hai_cham_dau_tien(e):
    if len(e) == 1:
        return e
    cond, colon = e[0], e.find(":")                      # SAI
    a, b = e[2:colon], e[colon + 1:]
    return sai_dau_hai_cham_dau_tien(a if cond == "T" else b)


for e in ["T?2:3", "F?1:T?4:5", "T?T?F:5:3"]:
    try:
        r = sai_dau_hai_cham_dau_tien(e)
    except Exception as ex:
        r = type(ex).__name__
    print(f"{e:<12} sai: {r!s:<12} | đúng: {Solution().parseTernary(e)}")

T?2:3        sai: 2            | đúng: 2
F?1:T?4:5    sai: 4            | đúng: 4
T?T?F:5:3    sai: IndexError   | đúng: F


**Vì sao sai:** ở `T?T?F:5:3` dấu `:` đầu tiên thuộc biểu thức **con** `T?F:5`, không phải dấu phân tách hai nhánh của biểu thức ngoài. Cắt ở đó ra nhánh đúng `"T?F"` (không hợp lệ) và nhánh sai `"5:3"`. **Cách sửa:** tìm dấu `:` **khớp** bằng biến đếm lồng (`depth` tăng ở `?`, giảm ở `:`), như mục 3.4 và Cách 5.

### Lỗi 2: Đảo nhánh khi chọn giá trị

In [28]:
def sai_dao_nhanh(expression):
    stk, cond = [], False
    for c in expression[::-1]:
        if c == ":":
            continue
        if c == "?":
            cond = True
        elif cond:
            x = stk.pop()                                # nhánh đúng
            y = stk.pop()                                # nhánh sai
            stk.append(y if c == "T" else x)             # SAI: ngược
            cond = False
        else:
            stk.append(c)
    return stk[0]


for e in ["T?2:3", "F?1:T?4:5", "T?T?F:5:3"]:
    print(f"{e:<12} sai: {sai_dao_nhanh(e)} | đúng: {Solution().parseTernary(e)}")

T?2:3        sai: 3 | đúng: 2
F?1:T?4:5    sai: 1 | đúng: 4
T?T?F:5:3    sai: 3 | đúng: F


**Vì sao sai:** duyệt từ **phải sang trái** nên nhánh sai `b` vào ngăn xếp **trước**, nhánh đúng `a` vào **sau** và nằm trên đỉnh. Nhầm thứ tự `pop` làm đảo kết quả. **Cách sửa:** `a = stk.pop()` (đúng), `b = stk.pop()` (sai); `T` ⇒ giữ `a`, `F` ⇒ giữ `b`. Hãy tự kiểm với `T?2:3` (phải ra `2`).

### Lỗi 3: Đệ quy không nâng giới hạn với biểu thức rất sâu

In [29]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)                                # giả lập giới hạn mặc định
deep = "T?" * 2000 + "1" + ":2" * 2000
try:
    print(parse_recursive(deep))
except RecursionError:
    print("RecursionError với đệ quy trần ở độ sâu 2000")
print("ngăn xếp (không đệ quy):", parse_stack_rtl(deep), "| nhảy (không đệ quy):", parse_jump(deep))
sys.setrecursionlimit(old if old > 20000 else 20000)       # khôi phục để không ảnh hưởng các cell sau

RecursionError với đệ quy trần ở độ sâu 2000
ngăn xếp (không đệ quy): 1 | nhảy (không đệ quy): 1


**Vì sao sai:** mỗi tầng lồng là một lời gọi đệ quy; biểu thức tới `10^4` ký tự có thể lồng khoảng `2500` tầng, vượt `~1000` mặc định của Python. **Cách sửa:** dùng ngăn xếp tường minh (Cách 3/4) hoặc vòng lặp nhảy (Cách 5), hoặc `sys.setrecursionlimit(...)` nếu vẫn chọn đệ quy.

### Lỗi 4: Rút gọn mẫu `X?a:b` đầu tiên từ **bên trái**

In [30]:
import re

def sai_rut_gon_tu_trai(expression):
    s = expression
    pattern = re.compile(r"([TF])\?(.):(.)")             # khớp mẫu 'X?a:b' xuất hiện ĐẦU TIÊN từ trái
    while True:
        m = pattern.search(s)
        if m is None:
            return s
        value = m.group(2) if m.group(1) == "T" else m.group(3)
        s = s[:m.start()] + value + s[m.end():]


for e in ["T?1:F?2:3", "T?2:T?3:4", "F?1:T?4:5"]:
    print(f"{e:<12} sai: {sai_rut_gon_tu_trai(e):<8} | đúng: {Solution().parseTernary(e)}")

T?1:F?2:3    sai: 1?2:3    | đúng: 1
T?2:T?3:4    sai: 2?3:4    | đúng: 2
F?1:T?4:5    sai: 4        | đúng: 4


**Vì sao sai:** ở `T?1:F?2:3` mẫu đầu tiên từ trái là `T?1:F`, trong đó `F` thật ra là **điều kiện của biểu thức kế tiếp** `F?2:3`, không phải nhánh sai. Rút gọn thành `1` rồi còn dư `?2:3` ⇒ chuỗi `"1?2:3"` vô nghĩa (đáp án đúng là `"1"`). Ở `F?1:T?4:5` ta may mắn ra đúng vì giá trị `T` được "tái sử dụng" làm điều kiện, nhưng không thể dựa vào may mắn. **Cách sửa:** luôn rút gọn từ **`?` ngoài cùng bên phải** (Cách 1), hoặc dùng ngăn xếp/nhìn trước để biết một ký tự là điều kiện hay giá trị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Biểu thức lồng, kết hợp phải→trái**" ⇒ duyệt **từ phải sang trái** với ngăn xếp: khi gặp toán tử thì hai toán hạng đã sẵn ở đỉnh.
- Nhận dạng điều kiện/giá trị bằng **vị trí** (ký tự đứng ngay trước `?` là điều kiện), **không** bằng loại ký tự: `T`/`F` có thể vừa là điều kiện vừa là giá trị.
- Có thể bỏ qua nhánh không chọn bằng **đếm lồng** `?`/`:` như khớp ngoặc (Cách 5): `O(1)` bộ nhớ phụ.
- Biểu thức rất sâu ⇒ cẩn thận giới hạn đệ quy; ưu tiên vòng lặp + ngăn xếp.
- Tự kiểm bằng **bộ sinh cây → chuỗi** có giá trị biết trước: kỹ thuật chung cho mọi bài parser/biểu thức.
- Cùng họ: *Basic Calculator (224, 227, 772)*, *Decode String (394)*, *Valid Parentheses (20)*, *Parse Lisp Expression (736)*.

## 15. Biến thể của bài

1. **Điều kiện là biểu thức ba ngôi có ngoặc** `(T?F:T)?1:2` — cần xử lý ngoặc.
2. **Giá trị nhiều chữ số** (số nguyên tuỳ ý) — cần tách token trước.
3. **Điều kiện là so sánh số** (ví dụ `3>2?1:0`) — cần thêm toán tử so sánh.
4. **Trả về biểu thức dạng cây/ngoặc đầy đủ** thay vì giá trị (in cách nhóm).
5. **Đếm số lần "nhánh được chọn" là lá** hoặc **độ sâu lồng tối đa**.
6. Bài liên quan: **Parse Lisp Expression (736)**, **Basic Calculator (224)**, **Number of Atoms (726)**.

In [31]:
def fully_parenthesize(expression):
    """Biến thể 4: viết lại biểu thức với ngoặc đầy đủ theo luật phải->trái (dùng ngăn xếp phải->trái)."""
    stk, cond = [], False
    for c in expression[::-1]:
        if c == ":":
            continue
        if c == "?":
            cond = True
        elif cond:
            a, b = stk.pop(), stk.pop()
            stk.append(f"({c}?{a}:{b})")
            cond = False
        else:
            stk.append(c)
    return stk[0]


def max_nesting(expression):
    """Biến thể 5: độ sâu lồng tối đa (số '?' lồng nhau tại một thời điểm khi đọc từ trái sang phải)."""
    depth = best = 0
    for c in expression:
        if c == "?":
            depth += 1
            best = max(best, depth)
        elif c == ":":
            depth -= 1
    return best


print(fully_parenthesize("F?1:T?4:5"))          # (F?1:(T?4:5))
print(fully_parenthesize("T?T?F:5:3"))          # (T?(T?F:5):3)
print(max_nesting("T?T?F:5:3"), max_nesting("F?1:F?2:F?3:4"))      # 2 1
assert fully_parenthesize("F?1:T?4:5") == "(F?1:(T?4:5))"
assert fully_parenthesize("T?T?F:5:3") == "(T?(T?F:5):3)"
assert max_nesting("T?T?F:5:3") == 2 and max_nesting("F?1:F?2:F?3:4") == 1

(F?1:(T?4:5))
(T?(T?F:5):3)
2 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Rút gọn chuỗi lặp lại | thay `X?a:b` ở `?` ngoài cùng phải | `O(n²)` | `O(n)` | dễ hiểu nhất, oracle |
| 2. Đệ quy theo con trỏ | định nghĩa đệ quy của ngữ pháp | `O(n)` | `O(d)` | khi cần mở rộng ngữ pháp (ngoặc, toán tử) |
| 3. Ngăn xếp phải→trái | toán hạng sẵn ở đỉnh khi gặp `?` | `O(n)` | `O(n)` | **nên nộp**, không đệ quy |
| 4. Ngăn xếp trái→phải | rút gọn khi đủ mẫu, có nhìn trước | `O(n)` | `O(n)` | khi buộc đọc theo chiều xuôi |
| 5. Nhảy theo điều kiện | chỉ đi vào nhánh được chọn | `O(n)` | `O(1)` | tiết kiệm bộ nhớ, đọc một lượt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"T?5:9"[::-1]`, `"F?1:T?4:5".rfind("?")`, `Solution().parseTernary("F?3:4")`, `Solution().parseTernary("T?F:T")`, `Solution().parseTernary("F?1:F?2:3")`.

**Bài 2 (tính tay).** Tính `T?F?1:2:F?3:4` bằng cách thêm ngoặc đầy đủ rồi rút gọn từng bước; sau đó lập bảng trace ngăn xếp phải→trái.

**Bài 3 (điền TODO).** Hoàn thành `parse_todo(expression)` theo Cách 3.

In [32]:
def parse_todo(expression):
    stk = []
    cond = False
    # TODO 1: for c in expression[::-1]:
    # TODO 2:     nếu c == ":" thì bỏ qua (continue)
    # TODO 3:     nếu c == "?" thì cond = True
    # TODO 4:     ngược lại nếu cond: a = stk.pop(); b = stk.pop(); đẩy a nếu c == "T" ngược lại b; cond = False
    # TODO 5:     ngược lại: đẩy c vào stk
    # TODO 6: return stk[0]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `parse_recursive_split(expression)` (đệ quy theo cách **cắt chuỗi**): với `e`, nếu `len(e) == 1` trả về `e`; ngược lại điều kiện `e[0]`, `e[1] == '?'`, dùng **đếm lồng** để tìm dấu `:` khớp rồi đệ quy vào **chỉ nhánh được chọn**.

**Bài 5 (biến thể / mini challenge).** (a) Viết `parse_with_parens(expression)` hỗ trợ **ngoặc** quanh biểu thức con ở bất kỳ nhánh nào, ví dụ `"(F?1:2)"` hoặc `"T?(F?1:2):3"` (điều kiện vẫn là `T`/`F` một ký tự; gợi ý: bỏ ngoặc bằng cách xử lý `(`/`)` khi đọc token). (b) Viết `count_true_branches(expression)` đếm số điều kiện `T` **được thực sự đánh giá** (tức nằm trên đường đi thực tế từ gốc, theo Cách 5).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"9:5?T"`, `5`, `"4"`, `"F"`, `"3"`. Bài 3: bám sát mục 9 và mục 6.
- **Mức 2 (vừa):** Bài 2: `T?(F?1:2):(F?3:4)` → `T?2:(F?3:4)` → `2`; hoặc theo luật: `T?F?1:2:F?3:4` = `T ? (F?1:2) : (F?3:4)` — đáp án `2`. Bài 4: `depth = 1`, quét từ `j = 2`: `?` → `depth += 1`, `:` → `depth -= 1`, dừng khi `depth == 0`; nhánh đúng là `e[2:j]`, nhánh sai là `e[j+1:]`.
- **Mức 3 (sâu):** Bài 5(a): cách dễ nhất là **tiền xử lý**: dùng ngăn xếp riêng để rút gọn từng cặp ngoặc từ trong ra ngoài bằng `Solution().parseTernary` rồi thay lại; hoặc viết đệ quy `parse()` có nhánh `if c == "(":`. 5(b): dùng Cách 5, mỗi lần gặp một điều kiện (kể cả khi nhảy qua nhánh) tăng đếm nếu điều kiện là `T`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert "T?5:9"[::-1] == "9:5?T"
assert "F?1:T?4:5".rfind("?") == 5
assert Solution().parseTernary("F?3:4") == "4"
assert Solution().parseTernary("T?F:T") == "F"
assert Solution().parseTernary("F?1:F?2:3") == "3"

# Bài 2
assert fully_parenthesize("T?F?1:2:F?3:4") == "(T?(F?1:2):(F?3:4))"
assert Solution().parseTernary("T?F?1:2:F?3:4") == "2"


# Bài 3
def parse_todo_ans(expression):
    stk = []
    cond = False
    for c in expression[::-1]:
        if c == ":":
            continue
        if c == "?":
            cond = True
        elif cond:
            a = stk.pop()
            b = stk.pop()
            stk.append(a if c == "T" else b)
            cond = False
        else:
            stk.append(c)
    return stk[0]


for expr, expected in examples:
    assert parse_todo_ans(expr) == expected


# Bài 4
def parse_recursive_split(e):
    if len(e) == 1:
        return e
    cond = e[0]
    depth, j = 1, 2
    while depth:
        if e[j] == "?":
            depth += 1
        elif e[j] == ":":
            depth -= 1
        j += 1
    colon = j - 1                       # j đang đứng sau dấu ':' khớp
    return parse_recursive_split(e[2:colon] if cond == "T" else e[colon + 1:])


for expr, expected in examples:
    assert parse_recursive_split(expr) == expected
assert parse_recursive_split("F?T?1:2:T?3:4") == "3"


# Bài 5
def parse_with_parens(expression):
    pos = 0
    s = expression

    def parse():
        nonlocal pos
        if s[pos] == "(":
            pos += 1
            v = parse()
            pos += 1                    # bỏ ')'
            c = v
        else:
            c = s[pos]
            pos += 1
        if pos >= len(s) or s[pos] != "?":
            return c
        pos += 1
        a = parse()
        pos += 1
        b = parse()
        return a if c == "T" else b

    return parse()


def count_true_branches(expression):
    n = len(expression)
    i = 0
    count = 0
    while True:
        if i + 1 >= n or expression[i + 1] != "?":
            return count
        if expression[i] == "T":
            count += 1
            i += 2
        else:
            depth, j = 1, i + 2
            while depth:
                depth += (expression[j] == "?") - (expression[j] == ":")
                j += 1
            i = j


assert parse_with_parens("(F?1:2)") == "2"
assert parse_with_parens("T?(F?1:2):3") == "2"
assert parse_with_parens("F?1:(T?4:5)") == "4"
assert count_true_branches("T?T?F:5:3") == 2
assert count_true_branches("F?1:T?4:5") == 1
assert count_true_branches("F?1:F?2:3") == 0
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (biểu thức kết hợp phải→trái: duyệt ngược + ngăn xếp)

```python
class Solution:
    def parseTernary(self, expression: str) -> str:
        stk = []
        cond = False
        for c in expression[::-1]:
            if c == ":":
                continue
            if c == "?":
                cond = True                  # ký tự tiếp theo là điều kiện
            elif cond:
                a = stk.pop()                # nhánh đúng
                b = stk.pop()                # nhánh sai
                stk.append(a if c == "T" else b)
                cond = False
            else:
                stk.append(c)
        return stk[0]
```

**Checklist**

- [ ] Hiểu luật kết hợp **phải → trái** (`a?b:c?d:e` = `a?b:(c?d:e)`)?
- [ ] Duyệt **từ phải sang trái**; `:` bỏ qua; `?` bật cờ điều kiện?
- [ ] Khi gặp điều kiện: `a = pop()` là nhánh **đúng**, `b = pop()` là nhánh **sai** (đúng thứ tự)?
- [ ] Phân biệt điều kiện và giá trị bằng **vị trí**, không bằng loại ký tự (`T`/`F` có thể là giá trị)?
- [ ] Nếu dùng đệ quy: đã tính tới độ sâu `~2500` và giới hạn đệ quy chưa?
- [ ] Đã thử `"T?2:3"` (`"2"`), `"F?1:T?4:5"` (`"4"`), `"T?T?F:5:3"` (`"F"`) và `"F?1:F?2:F?3:4"` (`"4"`)?
- [ ] Skeleton đúng `class Solution` / `def parseTernary(self, expression: str) -> str`?